In [1]:
import pandas as pd

In [2]:
import re
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd

ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"
MASTER_FILE = ROOT / "stations_master.csv"
OUT_FILE = ROOT / "combined" / "all_data_with_lat_lon.csv"
UNMATCHED_FILE = ROOT / "combined" / "unmatched_stations_review.csv"

def normalize(text: str) -> str:
    if text is None:
        return ""
    s = str(text).lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[_/\\-]+", " ", s)         # underscores/hyphens/slashes -> space
    s = re.sub(r"[().,]", " ", s)           # punctuation to space
    s = re.sub(r"[^a-z0-9\s]", " ", s)      # keep alnum + spaces
    s = re.sub(r"\s+", " ", s).strip()
    return s

def compact(text: str) -> str:
    return normalize(text).replace(" ", "")

def station_core(name: str) -> str:
    # Remove trailing agency suffix like " - APPCB", " - CPCB", etc.
    return re.sub(r"\s*-\s*[A-Za-z ]+$", "", str(name)).strip()

def sim(a: str, b: str) -> float:
    ca, cb = compact(a), compact(b)
    if not ca or not cb:
        return 0.0
    if ca == cb:
        return 1.0
    if ca in cb or cb in ca:
        return 0.92
    return SequenceMatcher(None, ca, cb).ratio()

def best_master_match(master_df: pd.DataFrame, state_name: str, city_name: str, station_name: str):
    st_in = state_name
    ct_in = city_name
    sn_in = station_name
    sn_in_core = station_core(station_name)

    best_idx = None
    best_score = -1.0

    for idx, row in master_df.iterrows():
        s_state = sim(st_in, row["state"])
        if s_state < 0.45:
            continue

        s_city = sim(ct_in, row["city"])

        s_station_full = sim(sn_in, row["station_name"])
        s_station_core = sim(sn_in_core, station_core(row["station_name"]))
        s_station = max(s_station_full, s_station_core)

        total = 0.45 * s_state + 0.25 * s_city + 0.30 * s_station

        if total > best_score:
            best_score = total
            best_idx = idx

    if best_idx is None or best_score < 0.68:
        return None, best_score

    return master_df.loc[best_idx], best_score

def collect_csv_files(data_dir: Path):
    return [p for p in data_dir.rglob("*.csv") if p.is_file()]

def main():
    if not MASTER_FILE.exists():
        raise FileNotFoundError(f"Master file not found: {MASTER_FILE}")
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

    master_df = pd.read_csv(MASTER_FILE)
    required_cols = {"station_id", "state", "city", "station_name", "latitude", "longitude"}
    missing = required_cols - set(master_df.columns)
    if missing:
        raise ValueError(f"stations_master.csv missing columns: {missing}")

    files = collect_csv_files(DATA_DIR)
    if not files:
        print("No CSV files found inside data folder.")
        return

    out_rows = []
    unmatched = []

    for f in files:
        rel = f.relative_to(DATA_DIR)
        parts = rel.parts

        # Expected pattern: state/city/station/year.csv
        if len(parts) < 4:
            continue

        state_folder = parts[0]
        city_folder = parts[1]
        station_folder = parts[2]
        year_file = parts[-1]

        match_row, score = best_master_match(master_df, state_folder, city_folder, station_folder)

        if match_row is None:
            unmatched.append({
                "source_file": str(rel),
                "state_folder": state_folder,
                "city_folder": city_folder,
                "station_folder": station_folder,
                "best_score": round(float(score), 3) if score == score else None
            })

        try:
            df = pd.read_csv(f, low_memory=False)
        except Exception as e:
            unmatched.append({
                "source_file": str(rel),
                "state_folder": state_folder,
                "city_folder": city_folder,
                "station_folder": station_folder,
                "best_score": None,
                "read_error": str(e)
            })
            continue

        df["state_folder"] = state_folder
        df["city_folder"] = city_folder
        df["station_folder"] = station_folder
        df["source_file"] = str(rel)
        df["year_file"] = year_file

        if match_row is not None:
            df["matched_station_id"] = match_row["station_id"]
            df["matched_state"] = match_row["state"]
            df["matched_city"] = match_row["city"]
            df["matched_station_name"] = match_row["station_name"]
            df["latitude"] = match_row["latitude"]
            df["longitude"] = match_row["longitude"]
            df["match_score"] = round(float(score), 3)
        else:
            df["matched_station_id"] = None
            df["matched_state"] = None
            df["matched_city"] = None
            df["matched_station_name"] = None
            df["latitude"] = None
            df["longitude"] = None
            df["match_score"] = round(float(score), 3) if score == score else None

        out_rows.append(df)

    if not out_rows:
        print("No valid CSV content to merge.")
        return

    combined_df = pd.concat(out_rows, ignore_index=True)

    OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(OUT_FILE, index=False)

    if unmatched:
        pd.DataFrame(unmatched).to_csv(UNMATCHED_FILE, index=False)

    print(f"Done. Rows written: {len(combined_df)}")
    print(f"Combined file: {OUT_FILE}")
    print(f"Unmatched review file: {UNMATCHED_FILE if unmatched else 'None'}")
    print(f"Unmatched station folders: {len(unmatched)}")

#if __name__ == "__main__":
    #main()

In [3]:
import pandas as pd
df = pd.read_csv("merged_data.csv")

df.count()

Timestamp              3552432
PM2.5 (µg/m³)          2874582
PM10 (µg/m³)           2573879
NO (µg/m³)             2957103
NO2 (µg/m³)            2980616
NOx (ppb)              3003335
NH3 (µg/m³)            1855931
SO2 (µg/m³)            2619257
CO (mg/m³)             2996072
Ozone (µg/m³)          2906122
Benzene (µg/m³)        2355341
Toluene (µg/m³)        2105516
Xylene (µg/m³)          997877
O Xylene (µg/m³)        171887
Eth-Benzene (µg/m³)     858104
MP-Xylene (µg/m³)       863060
AT (°C)                1341327
RH (%)                 2481436
WS (m/s)               2515595
WD (deg)               2497629
RF (mm)                1079781
TOT-RF (mm)            2279664
SR (W/mt2)             2399623
BP (mmHg)              1739046
VWS (m/s)              1130861
state                  3552432
city                   3552432
station_name           3552432
latitude               3552432
longitude              3552432
dtype: int64

In [6]:
files = collect_csv_files(ROOT / "combined")

total_rows = 0
failed_files = []

for f in files:
    try:
        for chunk in pd.read_csv(f, chunksize=100_000, low_memory=False):
            total_rows += len(chunk)
    except Exception as e:
        failed_files.append((str(f), str(e)))

print(f"Total CSV files found: {len(files)}")
print(f"Total combined rows across all CSV files: {total_rows}")

if failed_files:
    print(f"Files failed to read: {len(failed_files)}")
    for fp, err in failed_files[:10]:
        print(f"- {fp}: {err}")

Total CSV files found: 2863
Total combined rows across all CSV files: 24602472


In [8]:
# Use the existing `f` path variable (already set in your notebook)
row_count = 0
for chunk in pd.read_csv("merged_data.csv", chunksize=100_000, low_memory=False):
    row_count += len(chunk)

print(f"File: merged_data.csv")
print(f"Row count: {row_count}")

File: merged_data.csv
Row count: 3552432
